In [1]:
import pandas as pd
df = pd.read_csv("fpl_player_statistics.csv")
df.head
df.describe

<bound method NDFrame.describe of             player_name      club_name position_name  now_cost  total_points  \
0        Erling Haaland       Man City           FWD      14.7           239   
1     Bruno B.Fernandes        Man Utd           MID      10.4           235   
2       Gabriel Gabriel        Arsenal           DEF       7.3           209   
3       Antoine Semenyo       Man City           MID       8.0           202   
4    Morgan Gibbs-White  Nott'm Forest           MID       7.6           188   
..                  ...            ...           ...       ...           ...   
836        Jayden Danns      Liverpool           FWD       4.3             0   
837       Darwin Darwin      Liverpool           FWD       6.5             0   
838        Tyler Morton      Liverpool           MID       4.5             0   
839     James McConnell      Liverpool           MID       4.4             0   
840         Josh Gracey         Wolves           GKP       4.0             0   

     

In [2]:
import pandas as pd 
df2 = pd.read_csv("premier_league_players.csv")
df2.head

<bound method NDFrame.head of         Player Name  Age Market Value                    Club  \
0    Erling Haaland   24     €180.00m         Manchester City   
1       Bukayo Saka   23     €150.00m              Arsenal FC   
2     Florian Wirtz   22     €140.00m            Liverpool FC   
3       Cole Palmer   23     €120.00m              Chelsea FC   
4    Alexander Isak   25     €120.00m        Newcastle United   
..              ...  ...          ...                     ...   
494    Andrew Moran   21        €200k  Brighton & Hove Albion   
495       Karl Hein   23        €175k              Arsenal FC   
496    Lukas Nmecha   26        €150k            Leeds United   
497      James Hill   23        €150k         AFC Bournemouth   
498  Carl Rushworth   24        €100k  Brighton & Hove Albion   

               Position Nationality  
0        Centre-Forward      Norway  
1          Right Winger     England  
2    Attacking Midfield     Germany  
3    Attacking Midfield     England  

In [3]:
import requests
import pandas as pd
import os

API_KEY = "ba28390614fd4b4c99869f11513f147b"

headers = {"X-Auth-Token": API_KEY}

base_url = "https://api.football-data.org/v4"
competition = "PL"
season = 2025

url = "https://api.football-data.org/v4/competitions/PL"

response = requests.get(url, headers=headers)

print(response.status_code)

ConnectionError: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))

In [ ]:
url = f"{base_url}/competitions/{competition}/standings?season={season}"

response = requests.get(url, headers=headers)

standings = response.json()

standings.keys()

In [ ]:
table = standings["standings"][0]["table"]

df_standings = pd.json_normalize(table)

df_standings.head()

In [ ]:
df_standings.columns.tolist()

In [ ]:
df_standings = df_standings[
    [
        "position",
        "team.name",
        "playedGames",
        "won",
        "draw",
        "lost",
        "points",
        "goalsFor",
        "goalsAgainst",
        "goalDifference",
        "form"
    ]
]

In [ ]:
df_standings.columns = [
    "Position",
    "Team",
    "Played",
    "Wins",
    "Draws",
    "Losses",
    "Points",
    "Goals For",
    "Goals Against",
    "Goal Difference",
    "Recent Form"
]

In [ ]:
df_standings.head()

In [ ]:
df_standings.to_csv("premier_league_standings.csv", index=False)

In [ ]:
url = f"{base_url}/competitions/{competition}/matches?season={season}"

response = requests.get(url, headers=headers)

matches = response.json()

matches.keys()

In [ ]:
df_matches = pd.json_normalize(matches["matches"])
df_matches.columns.tolist()

In [ ]:
df_matches_clean = df_matches[
    [
        "id",
        "utcDate",
        "status",
        "matchday",
        "homeTeam.name",
        "awayTeam.name",
        "score.winner",
        "score.fullTime.home",
        "score.fullTime.away",
        "score.halfTime.home",
        "score.halfTime.away"
    ]
]

In [ ]:
df_matches_clean.columns = [
    "Match ID",
    "Date",
    "Status",
    "Matchday",
    "Home Team",
    "Away Team",
    "Winner",
    "Home Goals",
    "Away Goals",
    "Home HT Goals",
    "Away HT Goals"
]

In [ ]:
df_matches_clean.head()

In [ ]:
df_matches_clean.to_csv("premier_league_matches.csv", index=False)

In [ ]:
# Rename columns to match df
df2 = df2.rename(columns={
    "Player Name": "player_name",
    "Club": "club_name",
    "Position": "position_name",
    "Age": "age",
    "Market Value": "market_value",
    "Nationality": "nationality"
})
# Remove extra spaces
df["player_name"] = df["player_name"].str.strip()
df2["player_name"] = df2["player_name"].str.strip()

df["club_name"] = df["club_name"].str.strip()
df2["club_name"] = df2["club_name"].str.strip()
master_df = pd.merge(
    df,
    df2,
    on=["player_name", "club_name"],
    how="inner"
)

In [ ]:
print(master_df.shape)
master_df.head()

In [ ]:
unmatched = df[~df["player_name"].isin(master_df["player_name"])]

print(unmatched[["player_name", "club_name"]].head(20))

In [ ]:
club_mapping = {
    "Man City": "Manchester City",
    "Man Utd": "Manchester United",
    "Nott'm Forest": "Nottingham Forest",
    "West Ham": "West Ham United",
    "Wolves": "Wolverhampton Wanderers",
    "Bournemouth": "AFC Bournemouth",
    "Chelsea": "Chelsea FC",
    "Arsenal": "Arsenal FC",
    "Liverpool": "Liverpool FC",
    "Brighton": "Brighton & Hove Albion",
    "Newcastle": "Newcastle United",
    "Spurs": "Tottenham Hotspur",
    "Leicester": "Leicester City",
    "Leeds": "Leeds United"
}

df["club_name"] = df["club_name"].replace(club_mapping)

In [ ]:
master_df = pd.merge(
    df,
    df2,
    on=["player_name", "club_name"],
    how="inner"
)

print(master_df.shape)

In [ ]:
player_mapping = {
    "Bruno B.Fernandes": "Bruno Fernandes",
    "Gabriel Gabriel": "Gabriel Magalhães",
    "Marc Guéhi": "Marc Guehi",
    "Igor Thiago Thiago": "Igor Thiago",
    "João Pedro João Pedro": "João Pedro"
}

df["player_name"] = df["player_name"].replace(player_mapping)

In [ ]:
print(f"FPL players: {len(df)}")
print(f"Market value players: {len(df2)}")
print(f"Merged players: {len(master_df)}")

In [ ]:
master_df = pd.merge(
    df,
    df2,
    on=["player_name", "club_name"],
    how="inner"
)

print(f"FPL players: {len(df)}")
print(f"Market value players: {len(df2)}")
print(f"Merged players: {len(master_df)}")

In [ ]:
test_merge = pd.merge(
    df,
    df2,
    on="player_name",
    how="inner"
)

print(test_merge.shape)

In [ ]:
master_df = pd.merge(
    df,
    df2,
    on="player_name",
    how="inner",
    suffixes=("_fpl", "_market")
)

print(master_df.shape)
master_df[["player_name", "club_name_fpl", "club_name_market", "market_value"]].head()

In [ ]:
master_df.to_csv("master_player_dataset.csv", index=False)

In [ ]:
stars = [
    "Mohamed Salah",
    "Bukayo Saka",
    "Cole Palmer",
    "Alexander Isak",
    "Declan Rice",
    "Virgil van Dijk",
    "Phil Foden",
    "Rodri",
    "Martin Ødegaard",
    "Bruno Fernandes"
]

master_df[master_df["player_name"].isin(stars)][
    ["player_name", "club_name_fpl", "market_value"]
]

In [ ]:
missing_market = df2[
    ~df2["player_name"].isin(master_df["player_name"])
]

print(len(missing_market))
missing_market[["player_name", "club_name"]].head(30)

In [ ]:
missing_fpl = df[
    ~df["player_name"].isin(master_df["player_name"])
]

print(len(missing_fpl))
missing_fpl[["player_name", "club_name"]].head(30)

In [ ]:
player_mapping = {
    "Marc Guehi": "Marc Guéhi",
    "Virgil Virgil": "Virgil van Dijk",
    "Carlos Henrique Casemiro": "Casemiro",
    "Enzo Enzo": "Enzo Fernández",
    "Bruno Bruno G.": "Bruno Guimarães",
    "Matheus Matheus N.": "Matheus Nunes",
    "Jurriën J.Timber": "Jurrien Timber",
    "Enzo E.Le Fée": "Enzo Le Fée",
    "Dominic Calvert-Lewin": "Dominic Calvert Lewin",
    "Dango O.Dango": "Dango Ouattara",
    "Viktor Gyökeres": "Viktor Gyokeres",
    "Mohamed Salah": "Mohamed Salah",
    "Luis Díaz": "Luis Díaz",
    "Darwin Darwin": "Darwin Núñez",
    "Nicolas Jackson": "Nicolas Jackson",
    "Rodri": "Rodri"
}

df["player_name"] = df["player_name"].replace(player_mapping)
df2["player_name"] = df2["player_name"].replace(player_mapping)

In [ ]:
master_df = pd.merge(
    df,
    df2,
    on="player_name",
    how="inner",
    suffixes=("_fpl", "_market")
)

print(master_df.shape)

In [ ]:
missing_market = df2[
    ~df2["player_name"].isin(master_df["player_name"])
]

missing_market[[
    "player_name",
    "club_name"
]].sort_values("club_name")

In [ ]:
missing_fpl = df[
    ~df["player_name"].isin(master_df["player_name"])
]

missing_fpl[[
    "player_name",
    "club_name"
]].sort_values("club_name")

In [ ]:
stars = [
    "Mohamed Salah",
    "Erling Haaland",
    "Alexander Isak",
    "Bukayo Saka",
    "Cole Palmer",
    "Phil Foden",
    "Rodri",
    "Declan Rice",
    "Bruno Fernandes",
    "Martin Ødegaard",
    "Virgil van Dijk",
    "Bruno Guimarães",
    "Luis Díaz",
    "Enzo Fernández",
    "Josko Gvardiol",
    "Marc Guéhi",
    "Mohamed Kudus",
    "Nicolas Jackson",
    "Amad Diallo",
    "Darwin Núñez"
]

master_df[
    master_df["player_name"].isin(stars)
][["player_name","club_name_fpl","market_value"]].sort_values("player_name")

In [ ]:
player_mapping = {
    "Junior Kroupi Jr.": "Eli Junior Kroupi",
    "Bruno B.Fernandes": "Bruno Fernandes",
    "Gabriel Gabriel": "Gabriel Magalhães",
    "Marc Guéhi": "Marc Guehi",
    "Igor Thiago Thiago": "Igor Thiago",
    "João Pedro João Pedro": "João Pedro",
    "Carlos Henrique Casemiro": "Casemiro",
    "Virgil Virgil": "Virgil van Dijk",
    "Enzo Enzo": "Enzo Fernández",
    "Bruno Bruno G.": "Bruno Guimarães",
    "Matheus Matheus N.": "Matheus Nunes",
    "Jurriën J.Timber": "Jurrien Timber",
    "Enzo E.Le Fée": "Enzo Le Fée",
    "Dango O.Dango": "Dango Ouattara",
    "Rodrigo R.Gomes": "Rodrigo Gomes",
    "Toti Toti": "Toti",
    "Jhon J.Arias": "Jhon Arias"
}

df["player_name"] = df["player_name"].replace(player_mapping)

In [ ]:
master_df = pd.merge(
    df,
    df2,
    on="player_name",
    how="inner",
    suffixes=("_fpl", "_market")
)

print(master_df.shape)

In [ ]:
missing_market = df2[
    ~df2["player_name"].isin(master_df["player_name"])
]

missing_market = missing_market.sort_values(["club_name", "player_name"])

print(f"Remaining Transfermarkt players: {len(missing_market)}")

missing_market[["player_name", "club_name"]]

In [ ]:
missing_fpl = df[
    ~df["player_name"].isin(master_df["player_name"])
]

missing_fpl = missing_fpl.sort_values(["club_name", "player_name"])

print(f"Remaining FPL players: {len(missing_fpl)}")

missing_fpl[["player_name", "club_name"]]

In [ ]:
pd.set_option("display.max_rows", None)

missing_market = missing_market.sort_values(["club_name", "player_name"])

missing_market[["player_name", "club_name"]]

In [ ]:
player_mapping.update({
    "Mohamed M.Salah": "Mohamed Salah",
    "Micky van de Ven": "Micky van de Ven",
    "Pedro Porro": "Pedro Porro",
    "Junior Kroupi Jr.": "Eli Junior Kroupi",
    "Ben White": "Ben White",
    "Gabriel Jesus": "Gabriel Jesus",
    "Kaoru Mitoma": "Kaoru Mitoma",
    "Emiliano Martínez": "Emiliano Martínez",
    "Renato Veiga": "Renato Veiga",
    "Marc Guehi": "Marc Guéhi",
    "Alisson Becker": "Alisson",
    "Bernardo Silva": "Bernardo Silva",
    "Josko Gvardiol": "Josko Gvardiol",
    "Savinho": "Savinho",
    "Rodri": "Rodri",
    "Ruben Dias": "Rúben Dias",
    "Amad Diallo": "Amad Diallo",
    "Lisandro Martinez": "Lisandro Martínez",
    "Matthijs de Ligt": "Matthijs de Ligt"
})

In [ ]:
pd.set_option("display.max_rows", None)

missing_fpl = missing_fpl.sort_values(["club_name", "player_name"])

missing_fpl[["player_name", "club_name"]]

In [ ]:
player_mapping.update({
    "Enes Enes Ünal": "Enes Ünal",
    "Francisco Evanilson Evanilson": "Evanilson",
    "Hamed H.Traorè": "Hamed Junior Traoré",
    "Illia Zabarnyi": "Ilya Zabarnyi",
    "Julián J.Araujo": "Julián Araujo",
    "Junior Kroupi.Jr": "Eli Junior Kroupi",
    "Philip Philip": "Philip Billing",

    "Benjamin White": "Ben White",
    "Gabriel G.Jesus": "Gabriel Jesus",
    "Fábio Fábio Vieira": "Fábio Vieira",
    "Viktor Gyökeres": "Viktor Gyokeres",

    "Emiliano Martínez": "Emiliano Martínez",
    "Samuel Illing Jr": "Samuel Iling-Junior",
    "Tammy Abraham": "Tammy Abraham",
    "Victor Lindelöf": "Victor Lindelöf",
    "Álex Alex Moreno": "Álex Moreno",

    "Mohamed M.Salah": "Mohamed Salah",
    "Luis Luis Díaz": "Luis Díaz",
    "Curtis C.Jones": "Curtis Jones",
    "Endo Endo": "Wataru Endo",
    "Alisson Becker": "Alisson",

    "Bernardo Bernardo": "Bernardo Silva",
    "Ederson Ederson M.": "Ederson",
    "Joško Gvardiol": "Josko Gvardiol",
    "Rodrigo 'Rodri' Rodrigo": "Rodri",
    "Rúben Rúben": "Rúben Dias",
    "Sávio Savinho": "Savinho",
    "Vitor Vitor Reis": "Vitor Reis",

    "Amad Amad": "Amad Diallo",
    "Antony Antony": "Antony",
    "Diego D.Leon": "Diego León",
    "Lisandro Martínez": "Lisandro Martínez",
    "Matthijs de Ligt": "Matthijs de Ligt",

    "Jacob J.Murphy": "Jacob Murphy"
})

In [ ]:
df["player_name"] = df["player_name"].replace(player_mapping)

master_df = pd.merge(
    df,
    df2,
    on="player_name",
    how="inner",
    suffixes=("_fpl", "_market")
)

print(master_df.shape)

In [ ]:
master_df.to_csv("master_player_dataset.csv", index=False)

In [ ]:
missing_market = (
    df2[~df2["player_name"].isin(master_df["player_name"])]
    .sort_values(["club_name", "player_name"])
)

print(len(missing_market))
missing_market[["player_name", "club_name"]]

In [ ]:
player_mapping.update({
    "Renato Renato Veiga": "Renato Veiga",
    "Jan Paul Van Hecke": "Jan Paul van Hecke",
    "Igor Thiago Thiago": "Igor Thiago",
    "Mamadou Mamadou Sarr": "Mamadou Sarr",
    "Matthijs de Ligt": "Matthijs de Ligt",
    "Micky Micky van de Ven": "Micky van de Ven",
    "Pedro Pedro Porro": "Pedro Porro",
    "Lucas Lucas Paquetá": "Lucas Paquetá",
    "Richarlison Richarlison": "Richarlison"
})

In [ ]:
df["player_name"] = df["player_name"].replace(player_mapping)

master_df = pd.merge(
    df,
    df2,
    on="player_name",
    how="inner",
    suffixes=("_fpl", "_market")
)

print(master_df.shape)

In [ ]:
big_names = [
    "Mohamed Salah","Rodri","Rúben Dias","Bernardo Silva","Josko Gvardiol",
    "Savinho","Micky van de Ven","Pedro Porro","Lucas Paquetá",
    "Richarlison","Matthijs de Ligt","Gabriel Jesus","Ben White",
    "Marc Guéhi","Emiliano Martínez","Renato Veiga","Mitoma",
    "Alisson","Luis Díaz","Curtis Jones"
]

for name in big_names:
    if name not in master_df["player_name"].values:
        print(name)

In [ ]:
master_df = pd.merge(
    df,
    df2[["player_name", "market_value"]],
    on="player_name",
    how="left"
)

In [ ]:
master_df["market_value"].notna().sum()

In [ ]:
print(master_df.shape)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = master_df.copy()

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
cols_to_drop = [
    "news",
    "penalties_text",
    "corners_and_indirect_freekicks_text",
    "direct_freekicks_text",
    "penalties_order",
    "corners_and_indirect_freekicks_order",
    "direct_freekicks_order"
]

df = master_df.drop(columns=cols_to_drop)

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
master_df.shape

In [ ]:
df["player_name"].head(1000)

In [ ]:
duplicates = df[df.duplicated("player_name", keep=False)] \
    .sort_values("player_name")

duplicates[["player_name", "club_name", "position_name"]]

In [ ]:
duplicate_rows = df[df.duplicated(subset="player_name", keep=False)]

duplicate_rows.groupby("player_name").nunique().T

In [ ]:
df = (
    df.sort_values("market_value", na_position="last")
      .drop_duplicates(subset="player_name", keep="first")
      .reset_index(drop=True)
)

print(df.shape)

In [ ]:
#PHASE 2
print(f"Players: {len(df)}")
print(f"Features: {df.shape[1]}")

df.describe()

In [ ]:
import matplotlib.pyplot as plt 

df["position_name"].value_counts().plot(kind="bar")

plt.title("Number of Players by Position")
plt.xlabel("Position")
plt.ylabel("Count")
plt.show()

In [ ]:
club_counts = df["club_name"].value_counts()

club_counts.plot(figsize=(12,5), kind="bar")
plt.title("Players by Club")
plt.ylabel("Players")
plt.show

In [ ]:
import numpy as np
import pandas as pd

# Work from the cleaned dataframe
model_df = df.copy()

# Convert minutes into 90-minute matches
model_df["nineties"] = model_df["minutes"] / 90

# Avoid division by zero for players with no minutes
model_df["nineties"] = model_df["nineties"].replace(0, np.nan)

# Create additional per-90 playing-style features
model_df["goals_per_90"] = (
    model_df["goals_scored"] / model_df["nineties"]
)

model_df["assists_per_90"] = (
    model_df["assists"] / model_df["nineties"]
)

model_df["tackles_per_90"] = (
    model_df["tackles"] / model_df["nineties"]
)

model_df["recoveries_per_90"] = (
    model_df["recoveries"] / model_df["nineties"]
)

model_df["clearances_blocks_interceptions_per_90"] = (
    model_df["clearances_blocks_interceptions"] /
    model_df["nineties"]
)

# Replace undefined values for players with no minutes
new_per90_features = [
    "goals_per_90",
    "assists_per_90",
    "tackles_per_90",
    "recoveries_per_90",
    "clearances_blocks_interceptions_per_90"
]

model_df[new_per90_features] = (
    model_df[new_per90_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

model_df[new_per90_features].describe()

In [ ]:
feature_groups = {
    "attacking": [
        "goals_per_90",
        "expected_goals_per_90",
        "threat"
    ],

    "creation": [
        "assists_per_90",
        "expected_assists_per_90",
        "creativity"
    ],

    "overall_attacking_involvement": [
        "expected_goal_involvements_per_90",
        "influence"
    ],

    "defending": [
        "tackles_per_90",
        "recoveries_per_90",
        "clearances_blocks_interceptions_per_90",
        "defensive_contribution_per_90"
    ]
}

for group, features in feature_groups.items():
    print(f"{group.upper()}:")
    for feature in features:
        print(f"  - {feature}")

In [ ]:

metadata_columns = [
    "player_name",
    "club_name",
    "position_name",
    "minutes",
    "market_value"
]

# Flatten the feature groups into one master feature list
model_features = [
    feature
    for group in feature_groups.values()
    for feature in group
]

print(f"Metadata columns: {len(metadata_columns)}")
print(f"Model features: {len(model_features)}")
print(model_features)

In [ ]:
MINIMUM_MINUTES = 450

eligible_players = model_df[
    model_df["minutes"] >= MINIMUM_MINUTES
].copy()

print(f"Players before minutes filter: {len(model_df)}")
print(f"Players after minutes filter: {len(eligible_players)}")

In [ ]:
# Keep only columns that actually exist
available_features = [
    feature
    for feature in model_features
    if feature in eligible_players.columns
]

missing_features = [
    feature
    for feature in model_features
    if feature not in eligible_players.columns
]

print("Available features:")
print(available_features)

print("\nMissing features:")
print(missing_features)

# Metadata used when displaying results
player_information = eligible_players[
    metadata_columns
].reset_index(drop=True)

# Numerical feature matrix used for modeling
X = eligible_players[
    available_features
].copy()

# Clean unexpected infinite or missing values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

print("\nFeature matrix shape:", X.shape)
X.head()

In [ ]:
feature_variation = X.nunique().sort_values()

constant_features = feature_variation[
    feature_variation <= 1
].index.tolist()

print("Constant features:", constant_features)

if constant_features:
    X = X.drop(columns=constant_features)
    available_features = [
        feature
        for feature in available_features
        if feature not in constant_features
    ]

print("Updated feature matrix shape:", X.shape)

In [ ]:
correlation_matrix = X.corr().abs()

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

highly_correlated_pairs = []

for column in upper_triangle.columns:
    correlated_features = upper_triangle.index[
        upper_triangle[column] > 0.90
    ].tolist()

    for feature in correlated_features:
        highly_correlated_pairs.append(
            {
                "feature_1": feature,
                "feature_2": column,
                "correlation": upper_triangle.loc[feature, column]
            }
        )

correlated_df = pd.DataFrame(highly_correlated_pairs)

if correlated_df.empty:
    print("No feature pairs have correlation above 0.90.")
else:
    display(
        correlated_df.sort_values(
            "correlation",
            ascending=False
        )
    )

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=X.index
)

print("Scaled feature matrix shape:", X_scaled_df.shape)

X_scaled_df.describe().round(2)

## Phase 4: Feature Standardization

Before clustering, the selected features are standardized so that each variable contributes equally to the distance calculations used by K-Means. Standardization transforms each feature to have a mean of 0 and a standard deviation of 1.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=X.index
)

print("Feature matrix shape:", X_scaled.shape)

X_scaled.describe().round(2)

In [ ]:
summary = pd.DataFrame({
    "Mean": X_scaled.mean(),
    "Std Dev": X_scaled.std()
})

summary.round(3)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt

k_values = range(2, 11)

inertia = []
silhouette_scores = []

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_scaled)

    inertia.append(kmeans.inertia_)

    silhouette_scores.append(
        silhouette_score(X_scaled, labels)
    )

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    k_values,
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")

plt.title("Elbow Method")

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    k_values,
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")

plt.title("Silhouette Scores")

plt.grid(True)

plt.show()

In [ ]:
results = pd.DataFrame({
    "Clusters": list(k_values),
    "Inertia": inertia,
    "Silhouette Score": silhouette_scores
})

display(results)

best_k = results.loc[
    results["Silhouette Score"].idxmax(),
    "Clusters"
]

print(f"Recommended number of clusters: {best_k}")

In [ ]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_scaled)

eligible_players["cluster"] = cluster_labels

print("K-Means model trained successfully.")
print(f"Number of clusters: {best_k}")

In [ ]:
cluster_counts = (
    eligible_players["cluster"]
    .value_counts()
    .sort_index()
    .reset_index()
)

cluster_counts.columns = ["Cluster", "Players"]

display(cluster_counts)

In [ ]:
player_information["cluster"] = cluster_labels

display(player_information.head())

In [ ]:
cluster_summary = (
    eligible_players
    .groupby("cluster")[available_features]
    .mean()
    .round(2)
)

display(cluster_summary)

In [ ]:
for cluster in sorted(eligible_players["cluster"].unique()):

    print("=" * 60)
    print(f"Cluster {cluster}")

    sample = (
        eligible_players[
            eligible_players["cluster"] == cluster
        ][
            ["player_name", "club_name", "position_name"]
        ]
        .head(10)
    )

    display(sample)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

principal_components = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    principal_components,
    columns=["PC1", "PC2"],
    index=X_scaled.index
)

pca_df["cluster"] = cluster_labels

pca_df.head()

In [ ]:
explained_variance = pd.DataFrame({
    "Component": ["PC1", "PC2"],
    "Explained Variance": pca.explained_variance_ratio_
})

explained_variance["Cumulative Variance"] = (
    explained_variance["Explained Variance"].cumsum()
)

display(explained_variance.round(4))

In [ ]:
plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    pca_df["PC1"],
    pca_df["PC2"],
    c=pca_df["cluster"],
    cmap="tab10",
    alpha=0.7
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Visualization of Player Role Clusters")
plt.colorbar(scatter, label="Cluster")
plt.grid(True)
plt.show()

In [ ]:
pca_players = player_information.copy()

pca_players["PC1"] = pca_df["PC1"].values
pca_players["PC2"] = pca_df["PC2"].values
pca_players["cluster"] = cluster_labels

display(
    pca_players[
        [
            "player_name",
            "club_name",
            "position_name",
            "cluster",
            "PC1",
            "PC2"
        ]
    ].head()
)

In [ ]:
top_players = (
    eligible_players
    .sort_values("minutes", ascending=False)
    .groupby("cluster")
    .head(3)
)

top_player_indices = top_players.index

plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    pca_df["PC1"],
    pca_df["PC2"],
    c=pca_df["cluster"],
    cmap="tab10",
    alpha=0.6
)

for index in top_player_indices:
    if index in pca_players.index:
        plt.annotate(
            pca_players.loc[index, "player_name"],
            (
                pca_players.loc[index, "PC1"],
                pca_players.loc[index, "PC2"]
            ),
            fontsize=8
        )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Visualization with Selected Player Labels")
plt.colorbar(scatter, label="Cluster")
plt.grid(True)
plt.show()

In [ ]:
pca_loadings = pd.DataFrame(
    pca.components_.T,
    columns=["PC1", "PC2"],
    index=X_scaled.columns
)

display(
    pca_loadings
    .sort_values("PC1", key=abs, ascending=False)
    .round(3)
)

In [ ]:
cluster_positions = pd.crosstab(
    eligible_players["cluster"],
    eligible_players["position_name"]
)

display(cluster_positions)

In [ ]:
eligible_players["market_value_numeric"] = pd.to_numeric(
    eligible_players["market_value"]
    .astype(str)
    .str.replace("€", "", regex=False)
    .str.replace("m", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

In [ ]:
cluster_market_value = (
    eligible_players
    .groupby("cluster")["market_value_numeric"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

display(cluster_market_value)

In [ ]:
for cluster in sorted(eligible_players["cluster"].unique()):

    print("=" * 80)
    print(f"CLUSTER {cluster}")

    display(
        eligible_players[
            eligible_players["cluster"] == cluster
        ]
        .sort_values(
            "market_value_numeric",
            ascending=False
        )[[
            "player_name",
            "club_name",
            "position_name",
            "market_value"
        ]]
        .head(20)
    )

In [ ]:
cluster_names = {
    0: "Attacking Creators & Finishers",
    1: "Defensive Core Players",
    2: "Goalkeepers"
}

In [ ]:
cluster_names = {
    0: "Attacking Creators & Finishers",
    1: "Defensive Core Players",
    2: "Goalkeepers"
}

eligible_players["role"] = eligible_players["cluster"].map(cluster_names)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(X_scaled)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=eligible_players.index,
    columns=eligible_players.index
)

similarity_df.head()

In [ ]:
def recommend_replacements(player_name, top_n=5, same_position=True):
    matches = eligible_players[
        eligible_players["player_name"].str.lower() == player_name.lower()
    ]

    if matches.empty:
        return f"Player '{player_name}' was not found."

    player_index = matches.index[0]
    player_cluster = eligible_players.loc[player_index, "cluster"]
    player_position = eligible_players.loc[player_index, "position_name"]

    candidate_mask = (
        (eligible_players["cluster"] == player_cluster) &
        (eligible_players.index != player_index)
    )

    if same_position:
        candidate_mask &= (
            eligible_players["position_name"] == player_position
        )

    candidates = eligible_players[candidate_mask].copy()

    candidates["similarity_score"] = (
        similarity_df.loc[player_index, candidates.index].values
    )

    results = (
        candidates
        .sort_values("similarity_score", ascending=False)
        .head(top_n)
        .copy()
    )

    results["similarity_percent"] = (
        results["similarity_score"] * 100
    ).round(1)

    return results[
        [
            "player_name",
            "club_name",
            "position_name",
            "role",
            "similarity_percent",
            "market_value"
        ]
    ].reset_index(drop=True)

In [ ]:
recommend_replacements(
    "Erling Haaland",
    top_n=10,
    same_position=True
)

In [ ]:
recommend_replacements("Rodri", top_n=10)

In [ ]:
recommend_replacements("William Saliba", top_n=10)

In [ ]:
def recommend_replacements(
    player_name,
    top_n=5,
    same_position=True,
    max_market_value=None
):
    matches = eligible_players[
        eligible_players["player_name"].str.lower() == player_name.lower()
    ]

    if matches.empty:
        return f"Player '{player_name}' was not found."

    player_index = matches.index[0]
    player_cluster = eligible_players.loc[player_index, "cluster"]
    player_position = eligible_players.loc[player_index, "position_name"]

    candidate_mask = (
        (eligible_players["cluster"] == player_cluster) &
        (eligible_players.index != player_index)
    )

    if same_position:
        candidate_mask &= (
            eligible_players["position_name"] == player_position
        )

    if max_market_value is not None:
        candidate_mask &= (
            eligible_players["market_value_numeric"] <= max_market_value
        )

    candidates = eligible_players[candidate_mask].copy()

    if candidates.empty:
        return "No players matched the selected filters."

    candidates["similarity_score"] = (
        similarity_df.loc[player_index, candidates.index].values
    )

    results = (
        candidates
        .sort_values("similarity_score", ascending=False)
        .head(top_n)
        .copy()
    )

    results["similarity_percent"] = (
        results["similarity_score"] * 100
    ).round(1)

    return results[
        [
            "player_name",
            "club_name",
            "position_name",
            "role",
            "similarity_percent",
            "market_value"
        ]
    ].reset_index(drop=True)

In [ ]:
recommend_replacements(
    "Erling Haaland",
    top_n=10,
    same_position=True,
    max_market_value=60
)

In [ ]:
def recommend_replacements(
    player_name,
    top_n=5,
    same_position=True,
    max_market_value=None
):
    matches = eligible_players[
        eligible_players["player_name"].str.lower() == player_name.lower()
    ]

    if matches.empty:
        return f"Player '{player_name}' was not found."

    player_index = matches.index[0]
    selected_player = eligible_players.loc[player_index]

    player_cluster = selected_player["cluster"]
    player_position = selected_player["position_name"]
    player_value = selected_player["market_value_numeric"]

    candidate_mask = (
        (eligible_players["cluster"] == player_cluster) &
        (eligible_players.index != player_index)
    )

    if same_position:
        candidate_mask &= (
            eligible_players["position_name"] == player_position
        )

    if max_market_value is not None:
        candidate_mask &= (
            eligible_players["market_value_numeric"] <= max_market_value
        )

    candidates = eligible_players[candidate_mask].copy()

    if candidates.empty:
        return "No players matched the selected filters."

    candidates["similarity_score"] = (
        similarity_df.loc[player_index, candidates.index].values
    )

    candidates["similarity_percent"] = (
        candidates["similarity_score"] * 100
    ).round(1)

    candidates["value_difference"] = (
        candidates["market_value_numeric"] - player_value
    ).round(2)

    candidates["value_difference_display"] = candidates[
        "value_difference"
    ].apply(
        lambda value: (
            f"+€{value:.2f}m"
            if value > 0
            else f"-€{abs(value):.2f}m"
            if value < 0
            else "€0.00m"
        )
        if pd.notna(value)
        else "Unknown"
    )

    results = (
        candidates
        .sort_values("similarity_score", ascending=False)
        .head(top_n)
        .copy()
    )

    selected_player_summary = pd.DataFrame({
        "Player": [selected_player["player_name"]],
        "Club": [selected_player["club_name"]],
        "Position": [selected_player["position_name"]],
        "Role": [selected_player["role"]],
        "Market Value": [selected_player["market_value"]]
    })

    recommendation_table = results[
        [
            "player_name",
            "club_name",
            "position_name",
            "role",
            "similarity_percent",
            "market_value",
            "value_difference_display"
        ]
    ].copy()

    recommendation_table.columns = [
        "Player",
        "Club",
        "Position",
        "Role",
        "Similarity (%)",
        "Market Value",
        "Value Difference"
    ]

    return selected_player_summary, recommendation_table.reset_index(drop=True)

In [ ]:
selected_player, recommendations = recommend_replacements(
    "Erling Haaland",
    top_n=10
)

print("SELECTED PLAYER")
display(selected_player)

print("RECOMMENDED REPLACEMENTS")
display(recommendations)

In [ ]:
selected_player, recommendations = recommend_replacements(
    "Erling Haaland",
    top_n=10,
    same_position=True,
    max_market_value=40
)

print("SELECTED PLAYER")
display(selected_player)

print("RECOMMENDED REPLACEMENTS UNDER €40M")
display(recommendations)

In [ ]:
recommend_replacements("Mohamed Salah")

In [ ]:
recommend_replacements("Virgil van Dijk")

In [ ]:
from IPython.display import display, Markdown

def recommend_player_replacements(
    player_name,
    top_n=5,
    same_position=True,
    max_market_value=None
):

    matches = eligible_players[
        eligible_players["player_name"].str.lower() == player_name.lower()
    ]

    if matches.empty:
        print(f"Player '{player_name}' was not found.")
        return

    player_index = matches.index[0]
    selected_player = eligible_players.loc[player_index]

    player_cluster = selected_player["cluster"]
    player_position = selected_player["position_name"]
    player_value = selected_player["market_value_numeric"]

    candidate_mask = (
        (eligible_players["cluster"] == player_cluster) &
        (eligible_players.index != player_index)
    )

    if same_position:
        candidate_mask &= (
            eligible_players["position_name"] == player_position
        )

    if max_market_value is not None:
        candidate_mask &= (
            eligible_players["market_value_numeric"] <= max_market_value
        )

    candidates = eligible_players[candidate_mask].copy()

    if candidates.empty:
        print("No players matched the selected filters.")
        return

    candidates["similarity_score"] = (
        similarity_df.loc[player_index, candidates.index].values
    )

    candidates["similarity_percent"] = (
        candidates["similarity_score"] * 100
    ).round(1)

    candidates["value_difference"] = (
        candidates["market_value_numeric"] - player_value
    ).round(2)

    candidates["value_difference_display"] = candidates[
        "value_difference"
    ].apply(
        lambda value:
            f"+€{value:.2f}m" if pd.notna(value) and value > 0
            else f"-€{abs(value):.2f}m" if pd.notna(value) and value < 0
            else "€0.00m" if pd.notna(value)
            else "Unknown"
    )

    results = (
        candidates
        .sort_values("similarity_score", ascending=False)
        .head(top_n)
        .copy()
    )

    selected_player_summary = pd.DataFrame({
        "Player": [selected_player["player_name"]],
        "Club": [selected_player["club_name"]],
        "Position": [selected_player["position_name"]],
        "Role": [selected_player["role"]],
        "Market Value": [selected_player["market_value"]]
    })

    recommendation_table = results[
        [
            "player_name",
            "club_name",
            "position_name",
            "role",
            "similarity_percent",
            "market_value",
            "value_difference_display"
        ]
    ].copy()

    recommendation_table.columns = [
        "Player",
        "Club",
        "Position",
        "Role",
        "Similarity (%)",
        "Market Value",
        "Value Difference"
    ]

    display(Markdown("## Selected Player"))
    display(selected_player_summary)

    display(Markdown("## Recommended Replacements"))
    display(recommendation_table.reset_index(drop=True))

In [ ]:
recommend_player_replacements("Erling Haaland")

In [ ]:
recommend_player_replacements("Rodri")

In [ ]:
recommend_player_replacements(
    "Erling Haaland",
    top_n=10,
    max_market_value=40
)

In [ ]:
recommend_player_replacements("Bukayo Saka", top_n=5)

In [ ]:
def compare_players(player_one, player_two):
    player_one_match = eligible_players[
        eligible_players["player_name"].str.lower() == player_one.lower()
    ]

    player_two_match = eligible_players[
        eligible_players["player_name"].str.lower() == player_two.lower()
    ]

    if player_one_match.empty or player_two_match.empty:
        print("One or both players were not found.")
        return

    player_one_data = player_one_match.iloc[0]
    player_two_data = player_two_match.iloc[0]

    comparison_features = [
        feature for feature in [
            "goals_per_90",
            "assists_per_90",
            "expected_goals_per_90",
            "expected_assists_per_90",
            "tackles_per_90",
            "recoveries_per_90",
            "defensive_contribution_per_90"
        ]
        if feature in eligible_players.columns
    ]

    comparison = pd.DataFrame({
        "Statistic": comparison_features,
        player_one_data["player_name"]: [
            player_one_data[feature] for feature in comparison_features
        ],
        player_two_data["player_name"]: [
            player_two_data[feature] for feature in comparison_features
        ]
    })

    display(comparison.round(2))

    comparison.set_index("Statistic").plot(
        kind="bar",
        figsize=(12, 6)
    )

    plt.title(
        f"{player_one_data['player_name']} vs "
        f"{player_two_data['player_name']}"
    )
    plt.ylabel("Performance Value")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
compare_players("Erling Haaland", "Ollie Watkins")

In [ ]:
export_columns = [
    "player_name",
    "club_name",
    "position_name",
    "role",
    "cluster",
    "market_value",
    "market_value_numeric"
] + available_features

streamlit_data = eligible_players[
    [column for column in export_columns if column in eligible_players.columns]
].copy()

streamlit_data.to_csv(
    "player_replacement_data.csv",
    index=False
)

print(streamlit_data.shape)
streamlit_data.head()

In [ ]:
import joblib

joblib.dump(scaler, "scaler.pkl")
joblib.dump(kmeans, "kmeans.pkl")
joblib.dump(available_features, "model_features.pkl")

In [ ]:
# Show every unique club name alphabetically
clubs = sorted(df["club_name"].dropna().unique())

for club in clubs:
    print(club)